## Ordinal Encoding
Ordinal encoding consist in replacing the categories by integers from 1 to n (or 0 to n-1), where n is the number of distinct categories of the variable.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

In [2]:
# load dataset
file_url = "https://raw.githubusercontent.com/cmps460s26/cmps460-content/refs/heads/main/examples/03.dp/datasets/houseprice.csv"
data = pd.read_csv(
    file_url,
    usecols=["Neighborhood", "Exterior1st", "Exterior2nd", "SalePrice"],
)

data.head()

,Neighborhood,Exterior1st,Exterior2nd,SalePrice
0,CollgCr,VinylSd,VinylSd,208500
1,Veenker,MetalSd,MetalSd,181500
2,CollgCr,VinylSd,VinylSd,223500
3,Crawfor,Wd Sdng,Wd Shng,140000
4,NoRidge,VinylSd,VinylSd,250000


In [3]:
# let's have a look at how many labels each variable has

for col in data.columns:
    print(col, ": ", len(data[col].unique()), " labels")

Neighborhood :  25  labels
Exterior1st :  15  labels
Exterior2nd :  16  labels
SalePrice :  663  labels


In [4]:
# let's explore the unique categories
data["Neighborhood"].unique()

array(['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 'Somerst',
       'NWAmes', 'OldTown', 'BrkSide', 'Sawyer', 'NridgHt', 'NAmes',
       'SawyerW', 'IDOTRR', 'MeadowV', 'Edwards', 'Timber', 'Gilbert',
       'StoneBr', 'ClearCr', 'NPkVill', 'Blmngtn', 'BrDale', 'SWISU',
       'Blueste'], dtype=object)

In [5]:
data["Exterior1st"].unique()

array(['VinylSd', 'MetalSd', 'Wd Sdng', 'HdBoard', 'BrkFace', 'WdShing',
       'CemntBd', 'Plywood', 'AsbShng', 'Stucco', 'BrkComm', 'AsphShn',
       'Stone', 'ImStucc', 'CBlock'], dtype=object)

In [6]:
data["Exterior2nd"].unique()

array(['VinylSd', 'MetalSd', 'Wd Shng', 'HdBoard', 'Plywood', 'Wd Sdng',
       'CmentBd', 'BrkFace', 'Stucco', 'AsbShng', 'Brk Cmn', 'ImStucc',
       'AsphShn', 'Stone', 'Other', 'CBlock'], dtype=object)

### Split into Train and Test before encoding

We select which digit to assign to each category using the train set, and then use those mappings in the test set.

In [7]:
# let's separate into training and testing set

X_train, X_test, y_train, y_test = train_test_split(
    data.drop("SalePrice", axis=1),  # predictors
    data["SalePrice"],  # target
    test_size=0.3,  # percentage of obs in test set
    random_state=0,
)  # seed to ensure reproducibility

X_train.shape, X_test.shape

((1022, 3), (438, 3))

In [8]:
X_train.isnull().sum()

,0
Neighborhood,0
Exterior1st,0
Exterior2nd,0


## Ordinal Encoding with Scikit-learn

In [11]:
# let's find the categorical variables
# selects all columns in X_train that have object (string) dtype
cat_vars = X_train.select_dtypes(include="object").columns.tolist()
cat_vars

['Neighborhood', 'Exterior1st', 'Exterior2nd']

In [17]:
# let's set up the encoder

encoder = OrdinalEncoder()

In [18]:
# let's set up the column transformer
# Apply the encoder only to the categorical 
# columns in cat_vars
# keeps all other columns unchanged.
ct = ColumnTransformer(
    transformers=[("oe", encoder, cat_vars)],
    remainder="passthrough"
)


ct.set_output(transform="pandas")

ColumnTransformer(remainder='passthrough',
                  transformers=[('oe', OrdinalEncoder(),
                                 ['Neighborhood', 'Exterior1st',
                                  'Exterior2nd'])])

In [19]:
# train the encoder

ct.fit(X_train)

ColumnTransformer(remainder='passthrough',
                  transformers=[('oe', OrdinalEncoder(),
                                 ['Neighborhood', 'Exterior1st',
                                  'Exterior2nd'])])

In [20]:
# the categories that will be encoded in each variable

ct.named_transformers_["oe"].categories_

[array(['Blmngtn', 'Blueste', 'BrDale', 'BrkSide', 'ClearCr', 'CollgCr',
        'Crawfor', 'Edwards', 'Gilbert', 'IDOTRR', 'MeadowV', 'Mitchel',
        'NAmes', 'NPkVill', 'NWAmes', 'NoRidge', 'NridgHt', 'OldTown',
        'SWISU', 'Sawyer', 'SawyerW', 'Somerst', 'StoneBr', 'Timber',
        'Veenker'], dtype=object),
 array(['AsbShng', 'AsphShn', 'BrkComm', 'BrkFace', 'CBlock', 'CemntBd',
        'HdBoard', 'ImStucc', 'MetalSd', 'Plywood', 'Stone', 'Stucco',
        'VinylSd', 'Wd Sdng', 'WdShing'], dtype=object),
 array(['AsbShng', 'AsphShn', 'Brk Cmn', 'BrkFace', 'CBlock', 'CmentBd',
        'HdBoard', 'ImStucc', 'MetalSd', 'Other', 'Plywood', 'Stone',
        'Stucco', 'VinylSd', 'Wd Sdng', 'Wd Shng'], dtype=object)]

In [21]:
# transform data
X_train_enc = ct.transform(X_train)
X_test_enc = ct.transform(X_test)

X_train_enc.head()

,oe__Neighborhood,oe__Exterior1st,oe__Exterior2nd
64,5.0,12.0,13.0
682,4.0,13.0,14.0
960,3.0,13.0,10.0
1384,7.0,14.0,15.0
1100,18.0,13.0,14.0
